In [1]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv('data/merged/train_processed.csv')

# Define haversine function
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Radius of Earth in kilometers
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

data['haversine_distance'] = haversine(
    data['pickup_latitude_rounded'], data['pickup_longitude_rounded'],
    data['dropoff_latitude_rounded'], data['dropoff_longitude_rounded']
)

# Subset and clean data
subset = data[['haversine_distance', 'passenger_count', 'fare_amount', 'trip_duration']].dropna()
subset = subset[np.isfinite(subset).all(axis=1)]

# Remove outliers using IQR
Q1 = subset.quantile(0.25)
Q3 = subset.quantile(0.75)
IQR = Q3 - Q1
filtered_subset = subset[~((subset < (Q1 - 1.5 * IQR)) | (subset > (Q3 + 1.5 * IQR))).any(axis=1)]

In [3]:
# Prepare features and target
X = filtered_subset.drop(columns=['trip_duration'])
y = filtered_subset['trip_duration']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define parameter grid
param_grid = {
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10]
}

In [4]:
# Train Decision Tree with GridSearch
tree_regressor = DecisionTreeRegressor(random_state=42)
grid_search = GridSearchCV(tree_regressor, param_grid, cv=5, scoring='r2', verbose=1, n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best parameters and evaluation
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("Best Parameters:", grid_search.best_params_)
print("Best R2 (Validation):", grid_search.best_score_)
print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R2:", r2_score(y_test, y_pred))


Fitting 5 folds for each of 64 candidates, totalling 320 fits
Best Parameters: {'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 20}
Best R2 (Validation): 0.4607459583641142
Test MAE: 164.04794209400754
Test MSE: 48118.06320689619
Test R2: 0.4716909799409693
